# CertVIC Main-scale diffusion -- T4x2 template

Template for later Main-500 diffusion on free Kaggle T4 x2. Do not run this
until the remaining VLM controls and go/no-go gates pass. The notebook is
structured as two deterministic GPU workers: shard0 uses
`CUDA_VISIBLE_DEVICES=0`, shard1 uses `CUDA_VISIBLE_DEVICES=1`, with a
single-GPU sequential fallback.

Expected Main-500 planning anchors: CPU planning 10-30 min, diffusion on T4x2
about 6-8 hr, quality/detectability CPU 15-60 min, human review about 20-25 hr.
This template emits no local results when generated.

In [ ]:
# Configuration placeholders. Fill these in only after Main-500 is approved.
import json, os, subprocess, sys, time, zipfile
from pathlib import Path
import torch

CERTVIC_DIR = None
PLAN_INPUT = None          # directory containing scale/edit plan JSONL files
ADE20K_INPUT = None
OUTPUT_DIR = "/kaggle/working"
RUN_TAG = "main500_diffusion"
GPU_COUNT = torch.cuda.device_count()
print("torch.cuda.device_count() =", GPU_COUNT)
for i in range(GPU_COUNT):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {round(p.total_memory / (1024 ** 3), 2)} GiB")
print("Template only until paths are filled; no GPU work performed here.")

In [ ]:
# Deterministic T4x2 worker skeleton. Uses shard-level outputs, logs, resume checks, and merge.
from pathlib import Path
import json, os, subprocess, sys, time, zipfile

WORKER = Path(OUTPUT_DIR) / "certvic_diffusion_worker.py"
WORKER.write_text("""
import os, json, sys
shard = int(os.environ['SHARD'])
print('diffusion worker placeholder', {'shard': shard, 'cuda': os.environ.get('CUDA_VISIBLE_DEVICES')}, flush=True)
# Replace this placeholder with the existing certvic.edit.engines.batch_generate call once
# the Main-500 edit plan and approved model inputs are mounted.
""")

def launch(shard, visible):
    env = dict(os.environ, SHARD=str(shard), CUDA_VISIBLE_DEVICES=str(visible), PYTHONUNBUFFERED="1")
    log = open(Path(OUTPUT_DIR) / f"log_diffusion_main_scale_shard{shard}.txt", "a")
    return subprocess.Popen([sys.executable, "-u", str(WORKER)], env=env, stdout=log, stderr=subprocess.STDOUT), log

def run_template():
    if GPU_COUNT >= 2:
        print("Would launch shard0 on CUDA_VISIBLE_DEVICES=0 and shard1 on CUDA_VISIBLE_DEVICES=1.")
    else:
        print("WARNING: single-GPU fallback would run shard0 then shard1 on CUDA_VISIBLE_DEVICES=0.")
    print("Shard outputs would be generated_shard0.jsonl / generated_shard1.jsonl and merged deterministically.")
    print("Download zip would be diffusion_main_scale_T4x2_outputs.zip, excluding model cache/weights.")

run_template()